In [13]:
from sentence_transformers import SentenceTransformer, util
import pandas as pd
import numpy as np
from collections import defaultdict
import pymupdf
import re

In [2]:
# Load model — support Bahasa Indonesia
model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [47]:
# Load data lowongan
df = pd.read_csv("glints_jobs_detail.csv")

# menampilkan semua kolom
print(df.columns.to_list())

# mengecek jumlah data yang kolom judulnya kosong
print(df[df["Judul"].isna()].shape[0])

print(df.info())

['Judul', 'Industri', 'Tipe', 'Kategori', 'Tanggal_Post', 'Berlaku_Hingga', 'Perusahaan', 'Website_Perusahaan', 'Logo_Perusahaan', 'Alamat', 'Kota', 'Provinsi', 'Negara', 'Latitude', 'Longitude', 'Gaji_Min', 'Gaji_Max', 'Gaji_Currency', 'Gaji_Unit', 'Skills', 'Pengalaman_Bulan', 'Pendidikan', 'Deskripsi', 'Tentang_Perusahaan', 'Benefit', 'Apply_Langsung', 'Link']
113
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1324 entries, 0 to 1323
Data columns (total 27 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Judul               1211 non-null   object 
 1   Industri            1184 non-null   object 
 2   Tipe                1211 non-null   object 
 3   Kategori            1211 non-null   object 
 4   Tanggal_Post        1211 non-null   object 
 5   Berlaku_Hingga      1206 non-null   object 
 6   Perusahaan          1211 non-null   object 
 7   Website_Perusahaan  847 non-null    object 
 8   Logo_Perusahaan     1187 non

In [48]:
# data cleaning
# drop apabila judulnya kosong, karena itu kolom utama
df = df.dropna(subset=["Judul"])

# cek jumlah data kosong setelah drop
print(df[df["Judul"].isna()].shape[0])
print(df.isnull().sum())
# print(df.notnull().sum())


0
Judul                   0
Industri               27
Tipe                    0
Kategori                0
Tanggal_Post            0
Berlaku_Hingga          5
Perusahaan              0
Website_Perusahaan    364
Logo_Perusahaan        24
Alamat                119
Kota                    0
Provinsi                0
Negara                  0
Latitude                0
Longitude               0
Gaji_Min              236
Gaji_Max              236
Gaji_Currency         236
Gaji_Unit             237
Skills                  0
Pengalaman_Bulan      240
Pendidikan              0
Deskripsi               0
Tentang_Perusahaan    153
Benefit               385
Apply_Langsung          0
Link                    0
dtype: int64


In [49]:
# memilih kolom yang akan digunakan untuk embedding
kolom_pilihan = [
    "Judul", 
    "Industri", 
    "Tipe", 
    "Kategori",
    "Tanggal_Post",
    "Berlaku_Hingga",
    "Perusahaan",
    "Kota", 
    "Provinsi", 
    "Negara", 
    "Gaji_Min",
    "Gaji_Max",
    "Gaji_Currency",
    "Skills", 
    "Pengalaman_Bulan",
    "Pendidikan", 
    "Deskripsi",
    "Link"
]

df = df[kolom_pilihan]

df.isnull().sum()




Judul                 0
Industri             27
Tipe                  0
Kategori              0
Tanggal_Post          0
Berlaku_Hingga        5
Perusahaan            0
Kota                  0
Provinsi              0
Negara                0
Gaji_Min            236
Gaji_Max            236
Gaji_Currency       236
Skills                0
Pengalaman_Bulan    240
Pendidikan            0
Deskripsi             0
Link                  0
dtype: int64

In [50]:
# mengisi nilai kosong dengan default
df['Industri'] = df['Industri'].fillna("umum")
df['Berlaku_Hingga'] = df['Berlaku_Hingga'].fillna("-")
df['Gaji_Min'] = df['Gaji_Min'].fillna(0)
df['Gaji_Max'] = df['Gaji_Max'].fillna(0)
df['Gaji_Currency'] = df['Gaji_Currency'].fillna("-")
df['Pengalaman_Bulan'] = df['Pengalaman_Bulan'].fillna(0)

df.isnull().sum()

Judul               0
Industri            0
Tipe                0
Kategori            0
Tanggal_Post        0
Berlaku_Hingga      0
Perusahaan          0
Kota                0
Provinsi            0
Negara              0
Gaji_Min            0
Gaji_Max            0
Gaji_Currency       0
Skills              0
Pengalaman_Bulan    0
Pendidikan          0
Deskripsi           0
Link                0
dtype: int64

In [51]:
df.head()

,Judul,Industri,Tipe,Kategori,Tanggal_Post,Berlaku_Hingga,Perusahaan,Kota,Provinsi,Negara,Gaji_Min,Gaji_Max,Gaji_Currency,Skills,Pengalaman_Bulan,Pendidikan,Deskripsi,Link
0,Data Analyst,Information Technology and Services,CONTRACTOR,Data Analyst,2026-03-27T07:22:55.019Z,2026-04-27,Kodegiri,Jakarta Pusat,DKI Jakarta,ID,6000000.0,7250000.0,IDR,"Data Visualization, Microsoft Excel, SQL, Tabl...",36.0,professional certificate,"JOB DESCRIPTIONS : \n 1. Process, analyze, an...",https://glints.com/id/opportunities/jobs/data-...
1,Data Analyst,Retail,FULL_TIME,Data Analyst,2026-04-06T10:51:25.537Z,2026-05-07,PT. DANGHUAN TEKNOLOGI INDONESIA,Jakarta Utara,DKI Jakarta,ID,5750000.0,7500000.0,IDR,"Data Visualization, Microsoft Excel, Data Anal...",12.0,bachelor degree,Kualifikasi: \n Pendidikan minimal SMA/SMK/D3 ...,https://glints.com/id/opportunities/jobs/data-...
4,Data Analyst,Information Technology and Services,FULL_TIME,Data Analyst,2026-03-26T07:13:08.794Z,2026-04-26,HIGO,Jakarta Barat,DKI Jakarta,ID,5700000.0,7000000.0,IDR,"Python, Data Visualization, Data Analysis, Sta...",12.0,bachelor degree,Requirement: \n 1. Bachelor’s degree in Data ...,https://glints.com/id/opportunities/jobs/data-...
5,Data Analyst,Information Technology and Services,FULL_TIME,Data Analyst,2026-01-28T01:10:00.502Z,2026-04-29,PT. Berlian Sistem Informasi,Jakarta Timur,DKI Jakarta,ID,9100000.0,15200000.0,IDR,"Data Analysis, Power BI, Data Visualization, M...",36.0,bachelor degree,Perform data management activities in accordan...,https://glints.com/id/opportunities/jobs/data-...
6,Data Analyst,Apparel & Fashion,FULL_TIME,Data Analyst,2026-02-04T15:25:22.151Z,2026-04-25,itsmostly,Bandung Barat,Jawa Barat,ID,3000000.0,5000000.0,IDR,"Data Analysis, Microsoft Word, Analytical Skil...",12.0,professional certificate,- Menganalisa data produk & data penjualanan \...,https://glints.com/id/opportunities/jobs/data-...


In [53]:
print(df['Skills'][0])
print(df['Deskripsi'][0])

# df['Deskripsi'] = df['Deskripsi'].str.replace("\n", " ", regex=True)

Data Visualization, Microsoft Excel, SQL, Tableau, Data Analysis, Power BI, Microsoft SQL Server
JOB DESCRIPTIONS :  
 1. Process, analyze, and interpret data to support business decision-making
 
 2. Develop and maintain reports using Microsoft Excel (advanced level)
 
 3. Create data transformations using Power Query and advanced formulas
 
 4. Manage and optimize data using SQL and MS SSIS
 
 5. Support automation workflows using Power Automate / TMAPS
 
 6. Monitor data accuracy and ensure data integrity
 
 7. Provide insights based on automotive trends and related data
 
 8. Collaborate with stakeholders to understand reporting needs
 
 
 JOB REQUIREMENTS :  
 1. Minimum 2–3 years of experience as a Data Analyst or similar role
 
 2. Fluent in Microsoft Excel (Power Query, Pivot Table, Table, Macro)
 
 3. Strong proficiency in SQL and MS SSIS
 
 4. Experience handling Power Automate / TMAPS
 
 5. Adaptive and familiar with automotive terms and industry updates
 
 6. Willing to wor

In [54]:
# membuat text gabungan dari semua kolom yang akan digunakan untuk embedding

# cek hasil deskripsi
print(df['Deskripsi'][0])

df["teks_gabungan"] = (
    "Judul " + df["Judul"] + " " +
    "Industri " + df["Industri"] + " " +
    "Tipe Pekerjaan " + df["Tipe"] + " " +
    "Kategori " + df["Kategori"] + " " +
    "Tanggal Post " + df["Tanggal_Post"] + " " +
    "Berlaku Hingga " + df["Berlaku_Hingga"] + " " +
    "Perusahaan " + df["Perusahaan"] + " " +
    "Kota " + df["Kota"] + " " +
    "Provinsi " + df["Provinsi"] + " " +
    "Negara " + df["Negara"] + " " +
    "Gaji minimal " + df["Gaji_Min"].astype(str) + " " +
    "Gaji maksimal " + df["Gaji_Max"].astype(str) + " "+
    "Gaji Currency " + df["Gaji_Currency"] + " " +
    "Skill " + df["Skills"] + " " +
    "Pengalaman " + df["Pengalaman_Bulan"].astype(str) + " bulan " +
    "Pendidikan " + df["Pendidikan"] + " " +
    "Deskripsi " + df["Deskripsi"]
)

df['teks_gabungan'][0]

# simpan ke csv clean.
df.to_csv("glints_jobs_detail_clean.csv", index=False)



JOB DESCRIPTIONS :  
 1. Process, analyze, and interpret data to support business decision-making
 
 2. Develop and maintain reports using Microsoft Excel (advanced level)
 
 3. Create data transformations using Power Query and advanced formulas
 
 4. Manage and optimize data using SQL and MS SSIS
 
 5. Support automation workflows using Power Automate / TMAPS
 
 6. Monitor data accuracy and ensure data integrity
 
 7. Provide insights based on automotive trends and related data
 
 8. Collaborate with stakeholders to understand reporting needs
 
 
 JOB REQUIREMENTS :  
 1. Minimum 2–3 years of experience as a Data Analyst or similar role
 
 2. Fluent in Microsoft Excel (Power Query, Pivot Table, Table, Macro)
 
 3. Strong proficiency in SQL and MS SSIS
 
 4. Experience handling Power Automate / TMAPS
 
 5. Adaptive and familiar with automotive terms and industry updates
 
 6. Willing to work onsite at client office (Central Jakarta)
 
 7. Strong analytical thinking and problem-solving 

In [10]:

# ── TAHAP 1: Encode semua lowongan (lakukan sekali, simpan hasilnya) ──
print("Encoding lowongan...")
job_embeddings = model.encode(df["teks_gabungan"].tolist(), show_progress_bar=True)

# Simpan embedding agar tidak perlu encode ulang setiap kali
import numpy as np
np.save("job_embeddings.npy", job_embeddings)

Encoding lowongan...


Batches:   0%|          | 0/42 [00:00<?, ?it/s]

In [ ]:
# membaca dari file pdf 
doc = pymupdf.open("CV_linkedin.pdf")
text_semua = []

for page in doc:
    text_semua.append(page.get_text("text"))
    
doc.close()
# print(text_semua[:200])

def bersihkan_teks(teks: str) -> str:
    # Hapus unusual line terminators
    teks = re.sub(r'[\u2028\u2029\u0085]', ' ', teks)

    # Normalkan whitespace
    teks = re.sub(r'\s+', ' ', teks)

    # Hapus karakter non-printable
    teks = re.sub(r'[^\x20-\x7E\u00C0-\u024F\u1E00-\u1EFF]', '', teks)
    
    teks = teks.replace('\n', ' ')

    return teks.strip()

cleaned_text = bersihkan_teks(" ".join(text_semua))
print(cleaned_text[:500])



Contact (Mobile) 6289518789042 (Email) andreamusholin@gmail.com (LinkedIn) www.linkedin.com/in/ andrea-alfian- sah-putra-6b465b1a0 Social Media www.instagram.com/andalf.md www.facebook.com/andrea.alfian.7 Portfolio github.com/uchihamadara37 Top Skills Java Python React Typescript Golang Languages Indonesian (Native or Bilingual) English (Limited Working) Certifications Advanced Learning Algorithms Sequences, Time Series and Prediction Custom and Distributed Training with TensorFlow Natural Langu


In [19]:
user_input = cleaned_text
user_embedding = model.encode(user_input)

# ── TAHAP 3: Hitung similarity & ambil Top-5 ──
scores = util.cos_sim(user_embedding, job_embeddings)[0]
top5_idx = scores.topk(5).indices.tolist()

In [20]:
print("\n🎯 Top 5 Lowongan Paling Relevan:\n")
for rank, idx in enumerate(top5_idx, 1):
    row = df.iloc[idx]
    print(f"{rank}. {row['Judul']} — {row['Perusahaan']}")
    # print(f"   📍 {row['Lokasi']}")
    print(f"   💰 {row['Gaji_Min']:,} - {row['Gaji_Max']:,}")
    print(f"   🛠️  {row['Skills']}")
    print(f"   ⭐ Score: {scores[idx]:.2f}")
    print(f"   🔗 {row['Link']}\n")


🎯 Top 5 Lowongan Paling Relevan:

1. IT Java Developer — PT 360 Teknologi Indonesia
   💰 5,300,000.0 - 7,950,000.0
   🛠️  Microservis, Java, Python
   ⭐ Score: 0.56
   🔗 https://glints.com/id/opportunities/jobs/it-java-developer/58696ef3-670c-4bec-8481-8519414be124

2. Android Java Developer — PT. Deptech Digital Indonesia
   💰 8,000,000.0 - 12,000,000.0
   🛠️  Retrofit, MVVM, Swift, React Native, Kotlin, Mobile Development, TypeScript, iOS Development, Front-End Architecture, Android Development, Frontend Development, Restful API, HTML, Java, CSS
   ⭐ Score: 0.55
   🔗 https://glints.com/id/opportunities/jobs/android-java-developer/3e4cfa8d-cdd0-4158-9b21-16d8fdb12316

3. Mobile Developer — PT Sigma Global Teknologi
   💰 9,000,000.0 - 11,000,000.0
   🛠️  Kotlin, Java, Android Development, React Native, Mobile UI Design, Mobile Development, Restful API
   ⭐ Score: 0.54
   🔗 https://glints.com/id/opportunities/jobs/mobile-developer/a8913438-66c7-4320-b142-3173b4605385

4. Junior Softwar

### Kasus untuk Mencari Skill yang paling relevan
berdasarkan gaji

In [17]:
# Buat kolom gaji tengah
df["Gaji_Tengah"] = (df["Gaji_Min"] + df["Gaji_Max"]) / 2

# Explode skills (1 baris per skill)
df["Skills_List"] = df["Skills"].fillna("").str.split(", ")

# menampilkan kolom Skills_List
# print(df["Skills_List"].head())

df_exploded = df.explode("Skills_List")
df_exploded = df_exploded[df_exploded["Skills_List"] != ""]

df_exploded.head()

,Judul,Industri,Tipe,Kategori,Tanggal_Post,Berlaku_Hingga,Perusahaan,Website_Perusahaan,Logo_Perusahaan,Alamat,...,Pengalaman_Bulan,Pendidikan,Deskripsi,Tentang_Perusahaan,Benefit,Apply_Langsung,Link,teks_gabungan,Gaji_Tengah,Skills_List
0,Data Analyst,Information Technology and Services,CONTRACTOR,Data Analyst,2026-03-27T07:22:55.019Z,2026-04-27,Kodegiri,http://www.kodegiri.com,https://images.glints.com/unsafe/glints-dashbo...,"Jalan kenanga 126B, Sinduadi, Mlati, Sleman, Y...",...,36.0,professional certificate,"JOB DESCRIPTIONS : \n 1. Process, analyze, an...",Kodegiri starts as A software house then later...,NaN,True,https://glints.com/id/opportunities/jobs/data-...,"Data Analyst Data Visualization, Microsoft Exc...",6625000.0,Data Visualization
0,Data Analyst,Information Technology and Services,CONTRACTOR,Data Analyst,2026-03-27T07:22:55.019Z,2026-04-27,Kodegiri,http://www.kodegiri.com,https://images.glints.com/unsafe/glints-dashbo...,"Jalan kenanga 126B, Sinduadi, Mlati, Sleman, Y...",...,36.0,professional certificate,"JOB DESCRIPTIONS : \n 1. Process, analyze, an...",Kodegiri starts as A software house then later...,NaN,True,https://glints.com/id/opportunities/jobs/data-...,"Data Analyst Data Visualization, Microsoft Exc...",6625000.0,Microsoft Excel
0,Data Analyst,Information Technology and Services,CONTRACTOR,Data Analyst,2026-03-27T07:22:55.019Z,2026-04-27,Kodegiri,http://www.kodegiri.com,https://images.glints.com/unsafe/glints-dashbo...,"Jalan kenanga 126B, Sinduadi, Mlati, Sleman, Y...",...,36.0,professional certificate,"JOB DESCRIPTIONS : \n 1. Process, analyze, an...",Kodegiri starts as A software house then later...,NaN,True,https://glints.com/id/opportunities/jobs/data-...,"Data Analyst Data Visualization, Microsoft Exc...",6625000.0,SQL
0,Data Analyst,Information Technology and Services,CONTRACTOR,Data Analyst,2026-03-27T07:22:55.019Z,2026-04-27,Kodegiri,http://www.kodegiri.com,https://images.glints.com/unsafe/glints-dashbo...,"Jalan kenanga 126B, Sinduadi, Mlati, Sleman, Y...",...,36.0,professional certificate,"JOB DESCRIPTIONS : \n 1. Process, analyze, an...",Kodegiri starts as A software house then later...,NaN,True,https://glints.com/id/opportunities/jobs/data-...,"Data Analyst Data Visualization, Microsoft Exc...",6625000.0,Tableau
0,Data Analyst,Information Technology and Services,CONTRACTOR,Data Analyst,2026-03-27T07:22:55.019Z,2026-04-27,Kodegiri,http://www.kodegiri.com,https://images.glints.com/unsafe/glints-dashbo...,"Jalan kenanga 126B, Sinduadi, Mlati, Sleman, Y...",...,36.0,professional certificate,"JOB DESCRIPTIONS : \n 1. Process, analyze, an...",Kodegiri starts as A software house then later...,NaN,True,https://glints.com/id/opportunities/jobs/data-...,"Data Analyst Data Visualization, Microsoft Exc...",6625000.0,Data Analysis


In [20]:
# Hitung rata-rata gaji per skill + frekuensi kemunculan
skill_stats = df_exploded.groupby("Skills_List").agg(
    Rata_Gaji    = ("Gaji_Tengah", "mean"),
    Frekuensi    = ("Gaji_Tengah", "count"),
    Gaji_Max_Abs = ("Gaji_Max", "max")
).reset_index()

skill_stats.head(20)

,Skills_List,Rata_Gaji,Frekuensi,Gaji_Max_Abs
0,Tunning Process,NaN,0,NaN
1,.NET,8.976705e+06,22,30000000.0
2,.NET CORE,9.875000e+06,2,14000000.0
3,.netcore,8.275000e+06,1,9930000.0
4,2D Animation,9.500000e+06,2,15000000.0
5,2d/3d,3.525000e+06,1,3800000.0
6,3D Animation,7.000000e+06,3,15000000.0
7,3D Design,5.000000e+06,15,9000000.0
8,3D GIS,3.162500e+06,2,5790000.0
9,3D Max,7.750000e+06,1,8000000.0


In [21]:
# Filter: minimal muncul di 3 lowongan (hindari outlier)
skill_stats = skill_stats[skill_stats["Frekuensi"] >= 3]
skill_stats = skill_stats.sort_values("Rata_Gaji", ascending=False)

print("\n📊 Rata-rata Gaji per Skill (muncul di ≥3 lowongan):\n")
skill_stats.head(20)



📊 Rata-rata Gaji per Skill (muncul di ≥3 lowongan):



,Skills_List,Rata_Gaji,Frekuensi,Gaji_Max_Abs
113,CI/CD,1.519471e+07,21,255000000.0
245,Docker,1.362692e+07,26,255000000.0
513,Next.Js,1.324888e+07,18,255000000.0
211,Data Modeling,1.323000e+07,5,20000000.0
635,Rust,1.311271e+07,7,20000000.0
14,AI Systems,1.270000e+07,4,24000000.0
332,GraphQL,1.230000e+07,5,30000000.0
769,VB.NET,1.181250e+07,4,22000000.0
45,Amazon Web Services (AWS),1.168214e+07,14,30000000.0
315,GIT,1.137708e+07,24,255000000.0
